# Visualisation brute des .fit

In [ ]:
from garmin_fit_sdk import Decoder, Stream
import pandas as pd
import numpy as np

def parse_fit(filepath):
    stream = Stream.from_file(filepath)
    decoder = Decoder(stream)
    
    # Le SDK officiel est beaucoup plus tolérant sur les champs développeurs
    messages, errors = decoder.read()
    
    if errors:
        print(f"Erreurs (non bloquantes) : {len(errors)}")
        for e in errors[:3]: print(f"  - {e}")
    
    # Les messages sont un dict {type_name: [list of message dicts]}
    # On cherche les "record_mesgs" (données seconde par seconde)
    records = messages.get('record_mesgs', [])
    print(f"Records extraits : {len(records)}")
    
    df = pd.DataFrame(records)
    
    if df.empty:
        return df
    
    print(f"Colonnes disponibles : {df.columns.tolist()}")
    
    # --- SÉLECTION & RENOMMAGE ---
    rename_map = {
        'timestamp': 'timestamp',
        'heart_rate': 'heart_rate',
        'enhanced_speed': 'speed_ms',       # m/s
        'enhanced_altitude': 'altitude',    # mètres
        'cadence': 'cadence',               # rpm
        'power': 'power',                   # watts
        'vertical_oscillation': 'vert_osc', # mm
        'stance_time': 'gct',               # ms (Ground Contact Time)
        'step_length': 'stride_length',     # mm
        'distance': 'distance',             # mètres (cumulée)
    }
    
    existing = {k: v for k, v in rename_map.items() if k in df.columns}
    df_clean = df[existing.keys()].rename(columns=existing).copy()
    
    # Conversions
    if 'speed_ms' in df_clean.columns:
        df_clean['speed_kmh'] = df_clean['speed_ms'] * 3.6
        df_clean['pace_min_km'] = np.where(
            df_clean['speed_ms'] > 0.5,  # Seuil pour éviter division par ~0
            1000 / (df_clean['speed_ms'] * 60),  # min/km
            np.nan
        )
    
    # Timestamp en index
    if 'timestamp' in df_clean.columns:
        df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
        df_clean = df_clean.set_index('timestamp')
    
    # --- HRV / RR ---
    hrv_mesgs = messages.get('hrv_mesgs', [])
    rr_all = []
    for hrv_msg in hrv_mesgs:
        rr_values = hrv_msg.get('time', [])
        if isinstance(rr_values, (list, tuple)):
            rr_all.extend([v for v in rr_values if v is not None and 0.2 < v < 3.0])
        elif rr_values is not None and 0.2 < rr_values < 3.0:
            rr_all.append(rr_values)
    
    df_rr = pd.DataFrame({'rr_s': rr_all}) if rr_all else pd.DataFrame()
    print(f"Intervalles RR : {len(df_rr)} points")
    
    return df_clean, df_rr

# --- EXECUTION ---
df_main, df_rr = parse_fit('../data/raw/hrv.fit')

print("\n--- Stats principales ---")
print(df_main.describe().T[['count', 'mean', 'min', 'max']])

print("\n--- Aperçu (début) ---")
print(df_main.head())

print("\n--- Aperçu (fin) ---")
print(df_main.tail())

print(f"\n--- Durée totale : {(df_main.index[-1] - df_main.index[0])} ---")

In [ ]:
# --- DIAGNOSTIC : Où se cachent les RR / HRV ? ---
from garmin_fit_sdk import Decoder, Stream

stream = Stream.from_file('../data/raw/hrv.fit')
decoder = Decoder(stream)
messages, errors = decoder.read()

# 1. Lister TOUS les types de messages disponibles dans le fichier
print("=== Types de messages disponibles ===")
for key in sorted(messages.keys()):
    print(f"  {key}: {len(messages[key])} messages")

# 2. Vérifier les developer_fields (Coros y met souvent ses données propriétaires)
print("\n=== Inspection des developer_fields (premier record avec dev fields) ===")
records = messages.get('record_mesgs', [])
for rec in records[:50]:
    dev = rec.get('developer_fields', {})
    if dev:
        print(f"  Timestamp: {rec.get('timestamp')}")
        for k, v in dev.items():
            print(f"    - {k}: type={type(v).__name__}, value={v}")
        break

# 3. Chercher spécifiquement tout ce qui ressemble à HRV/RR
print("\n=== Recherche de champs HRV/RR dans tous les messages ===")
hrv_keywords = ['hrv', 'rr', 'heart_rate_variability', 'rmssd', 'sdnn', 'beat']
found_any = False
for msg_type, msg_list in messages.items():
    if msg_list:
        sample = msg_list[0]
        for key in sample.keys():
            if any(kw in key.lower() for kw in hrv_keywords):
                print(f"  TROUVÉ: {msg_type} -> champ '{key}' = {sample[key]}")
                found_any = True

if not found_any:
    print("  Aucun champ HRV/RR trouvé.")
    print("  => Coros n'exporte probablement pas les RR dans ce .fit.")
    print("  => Vérifier si 'HRV Logging' est activé sur la montre.")